# Namma Space — Autonomous 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Namma Space Challenge (Round 1)**

This notebook trains a photorealistic 3D Gaussian Splatting digital twin directly from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### ⚡ 2-Step Quick Start:
1. **GPU Check**: In the top menu, go to **Runtime → Change runtime type** → select **T4 GPU** → click **Save**.
2. **Run Step 1**: Installs dependencies and verified CUDA rasterizers (~1 minute).
3. **Upload & Run Step 2**: Upload your room walkthrough video (`.mp4`) and run Step 2. It will automatically extract frames, solve camera poses, train 3DGS (30,000 iterations), and download `namma_space_assets.zip`!

### Step 1: One-Time Environment & CUDA Rasterizer Setup (~1 min)
Prepares PyTorch, pins legacy setuptools to avoid metadata compilation errors, and installs the CUDA rasterizers.

In [ ]:
import os, sys

print("=== Step 1/2: Initializing Namma Space Cloud GPU Environment ===")

# 1. Ensure GPU is connected
!nvidia-smi

# 2. Pin setuptools to bypass setuptools v70+ pyproject metadata bug
!pip install -q "setuptools<70.0.0" wheel ninja plyfile tqdm pycolmap typer rich opencv-python-headless scipy

# 3. Clone Roomsplat and Inria Gaussian Splatting
%cd /content
if not os.path.exists("/content/roomsplat"):
    !git clone -q https://github.com/Ahmedn1/roomsplat.git
if not os.path.exists("/content/gaussian-splatting"):
    !git clone -q --recursive https://github.com/graphdeco-inria/gaussian-splatting.git

%cd /content/roomsplat
!pip install -q -e .

# 4. Install CUDA Rasterizers (diff-gaussian-rasterization & simple-knn)
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"Installing CUDA rasterizers for Python {py_tag}...")

# Try matching precompiled wheels first (instant)
!pip install -q https://github.com/camenduru/diff-gaussian-rasterization/releases/download/v1.0/diff_gaussian_rasterization-0.0.0-{py_tag}-{py_tag}-linux_x86_64.whl || true
!pip install -q https://github.com/camenduru/simple-knn/releases/download/v1.0/simple_knn-0.0.0-{py_tag}-{py_tag}-linux_x86_64.whl || true

# If not installed, compile with legacy build hooks (-e)
try:
    import diff_gaussian_rasterization, simple_knn
except ImportError:
    print("Compiling rasterizers with CUDA (~45 seconds)...\n")
    %cd /content/gaussian-splatting
    !git submodule update --init --recursive
    !pip install --no-build-isolation -e submodules/diff-gaussian-rasterization
    !pip install --no-build-isolation -e submodules/simple-knn

# 5. Final verification
import torch, diff_gaussian_rasterization, simple_knn, pycolmap
print(f"\n✓ PyTorch {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print("✓ diff_gaussian_rasterization: Ready")
print("✓ simple_knn: Ready")
print("✓ pycolmap: Ready")
print("\n=================================================================")
print("  SUCCESS: Namma Space Cloud Environment is 100% Ready!")
print("  Upload your video to /content/ and proceed to Step 2 below.")
print("=================================================================")

### Step 2: Autonomous Reconstruction Pipeline (Video → Poses → 3DGS → Splat Download)
Automatically discovers your uploaded `.mp4`/`.mov` walkthrough video, extracts sharp frames, solves 3D camera poses via COLMAP, trains 30,000 3DGS iterations, and downloads `namma_space_assets.zip` directly to your laptop.

In [ ]:
import os, sys, glob, shutil
from pathlib import Path
from google.colab import files

# 1. Auto-detect uploaded video in /content
video_candidates = [
    f for f in glob.glob("/content/*")
    if f.lower().endswith((".mp4", ".mov", ".m4v")) and not f.startswith("/content/sample_data")
]

if not video_candidates:
    print("No video found in /content. Please select your room walkthrough video to upload:")
    uploaded = files.upload()
    video_candidates = list(uploaded.keys())
    if not video_candidates:
        raise FileNotFoundError("No video provided. Please upload an MP4/MOV room walkthrough.")

video_file = Path(video_candidates[0]).resolve()
video_size_mb = video_file.stat().st_size / (1024 * 1024)
print(f"\nTarget Video: {video_file.name} ({video_size_mb:.2f} MB)")

# Define output directories
out_dir = Path(f"/content/{video_file.stem}_splat")
sparse_dir = out_dir / "sparse" / "0"
viewer_dir = out_dir / "viewer"
viewer_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# STAGE 1: Frame Extraction & Camera Pose Estimation (COLMAP)
# =============================================================================
if not (sparse_dir / "cameras.bin").exists():
    print("\n--- STAGE 1: Extracting Frames & Solving Camera Trajectory ---")
    %cd /content/roomsplat
    !python -m roomsplat.cli run "{video_file}" \
        --name "Namma Space Digital Twin" \
        --fps 1.5 \
        --max-dim 1600 \
        --skip-train \
        --no-view
else:
    print(f"\nReusing existing camera poses in: {sparse_dir}")

# =============================================================================
# STAGE 2: 3D Gaussian Splatting Training (30,000 Iterations)
# =============================================================================
print("\n--- STAGE 2: Training 3D Gaussian Radiance Field (30k Iterations on T4 GPU) ---")
%cd /content/gaussian-splatting

!python train.py \
    -s "{out_dir}" \
    -m "{out_dir}/model" \
    --iterations 30000 \
    --disable_viewer \
    --densify_until_iter 8000 \
    --densification_interval 200

# =============================================================================
# STAGE 3: Convert to 32-Byte scene.splat & Export cameras.json
# =============================================================================
print("\n--- STAGE 3: Packaging Web-Optimized Binary 3D Twin ---")
%cd /content/roomsplat
sys.path.insert(0, "/content/roomsplat")
from roomsplat.pipeline.export import ply_to_splat, write_cameras_json

ply_files = list(out_dir.glob("model/point_cloud/iteration_*/point_cloud.ply"))
if not ply_files:
    raise FileNotFoundError(f"point_cloud.ply not found in {out_dir}/model! Did training complete?")

ply_path = sorted(ply_files, key=lambda p: int(p.parent.name.split('_')[1]))[-1]
print(f"Found trained point cloud: {ply_path}")

splat_path = viewer_dir / "scene.splat"
ply_to_splat(ply_path, splat_path)
write_cameras_json(sparse_dir, viewer_dir)

# Stage for download
shutil.copy(splat_path, "/content/scene.splat")
shutil.copy(viewer_dir / "cameras.json", "/content/cameras.json")
!zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json

# =============================================================================
# STAGE 4: Automated Download to Local Machine
# =============================================================================
print("\n=================================================================")
print("  SUCCESS: 3D Twin Generation Complete!")
print("  Downloading namma_space_assets.zip to your computer...")
print("=================================================================\n")
files.download("/content/namma_space_assets.zip")